# Phase 3: Aircraft Operational Timeline and Rotation Reconstruction
## Flight Delays Are Contagious: Predicting Delay Propagation Across Consecutive Aircraft Operations

**Objective:** Build a unified UTC-anchored aircraft operational timeline and reconstruct valid consecutive flight pairs (`Flight_prev -> Flight_next`) operated by the same aircraft, enforcing sequence constraints, diversion sequence-breakers, and leak-free prediction-time feature availability.

**Core Pillars of Phase 3:**
1. **UTC Timestamp Normalization:** Conversion of local scheduled and actual timestamps to true Universal Coordinated Time (UTC) using verified IANA timezone mappings.
2. **Diversion Sequence Breaking:** Integration of 19,258 diverted operations as sequence-breaking barriers, successfully blocking 1,352 false completed-flight linkages.
3. **Physical Sequence Continuity:** Enforcement of airframe identity, station continuity ($DEST_{prev} == ORIGIN_{next}$), and chronological ordering ($ARR\_UTC_{prev} \le DEP\_UTC_{next}$).
4. **Preservation of Delayed Rotations:** Preserving 382,937 operations where incoming arrival delay consumed the scheduled turnaround window ($NEXT\_CRS\_DEP < PREV\_ARR$), capturing true delay propagation.
5. **Prediction Moment ($t_{pred}$):** Strict feature partitioning ensuring zero future target leakage.

In [ ]:
import os
import duckdb
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)

con = duckdb.connect()
print("Environment initialized successfully.")

### 1. Unified Operational Timeline Overview

In [ ]:
timeline_path = '../data/processed/aircraft_timeline.parquet'
q_timeline_summary = f"""
SELECT 
    COUNT(*) AS total_operations,
    SUM(CASE WHEN IS_DIVERTED = 0 THEN 1 ELSE 0 END) AS completed_operations,
    SUM(CASE WHEN IS_DIVERTED = 1 THEN 1 ELSE 0 END) AS diverted_operations,
    COUNT(DISTINCT TAIL_NUM) AS unique_aircraft,
    MIN(DEP_UTC) AS min_dep_utc,
    MAX(DEP_UTC) AS max_dep_utc
FROM read_parquet('{timeline_path}')
"""
df_tl_summary = con.execute(q_timeline_summary).df()
df_tl_summary

### 2. Operational Sequence and Rejection Reasons Breakdown

In [ ]:
df_rejections = pd.read_csv('../results/tables/rotation_rejection_reasons.csv')
df_rejections

### 3. Reconstructed Flight Rotation Pairs Summary

In [ ]:
df_summary = pd.read_csv('../results/tables/rotation_reconstruction_summary.csv')
df_summary

### 4. Turnaround Ground-Time Sensitivity Horizons

In [ ]:
df_sens = pd.read_csv('../results/tables/turnaround_sensitivity_analysis.csv')
df_sens

### 5. Rotation Invariants and Physical Feasibility Validation

In [ ]:
df_val = pd.read_csv('../results/tables/rotation_validation_report.csv')
df_val

### 6. Concrete Traceable Aircraft Rotation Sequence: Delay Propagation in Action

In [ ]:
pairs_path = '../data/processed/flight_rotation_pairs.parquet'
# Trace real aircraft N101NN across consecutive flights
q_trace = f"""
SELECT 
    PREV_FL_DATE,
    PREV_CARRIER,
    PREV_FL_NUM,
    PREV_ORIGIN,
    TURNAROUND_AIRPORT,
    NEXT_DEST,
    NEXT_FL_NUM,
    PREV_ARR_DELAY,
    SCHEDULED_TURNAROUND_BUFFER_MINUTES AS SCHED_BUFFER,
    ESTIMATED_REMAINING_TURNAROUND_MINUTES AS REMAINING_BUFFER,
    IS_TURNAROUND_DEFICIT,
    ACTUAL_GROUND_TIME_MINUTES AS RETRO_GROUND_TIME,
    NEXT_ARR_DELAY,
    NEXT_ARR_DEL15
FROM read_parquet('{pairs_path}')
WHERE TAIL_NUM = 'N101NN'
LIMIT 8
"""
df_trace = con.execute(q_trace).df()
df_trace

### 7. Feature Leakage Prevention Audit

In [ ]:
df_leakage = pd.read_csv('../results/tables/feature_leakage_validation.csv')
df_leakage

### 8. Phase 3 Conclusion and Transition to Phase 4

**Summary of Phase 3 Outcomes:**
1. Successfully normalized all 6,898,741 flight operations into UTC using validated IANA timezones.
2. Reconstructed **6,768,487 valid consecutive aircraft rotation pairs** ($98.11\%$ of operational timeline).
3. Blocked **19,109 operations** affected by diversions, preventing 1,352 false completed-flight connections.
4. Preserved **382,937 delayed rotations** where incoming arrival delay exceeded the scheduled departure time.
5. Empirically demonstrated that a **Turnaround Deficit** yields a **99.50% subsequent delay rate** (vs 17.38% with no deficit).
6. Verified 100% compliance with prediction-time availability rules (Zero negative ground times, zero leakage).

**Ready for Phase 4:** Exploratory Data Analysis (EDA) of general delay patterns and aircraft-level propagation dynamics.